### Import and Loading

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor
import joblib


In [2]:
## Paths
ROOT = Path.cwd().parents[0]          # if notebook is in week_02/
DATA_PATH = ROOT / "Data" / "cleaned_timeseries.csv"
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

### Loads + Basic Preparation

In [3]:
df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").set_index("date")

print("Loaded:", DATA_PATH)
print("Date range:", df.index.min(), "to", df.index.max(), "| rows:", len(df))
print("Target missing:", int(df["unit_sales"].isna().sum()))

Loaded: D:\data analytics\TimeSeries\Data\cleaned_timeseries.csv
Date range: 2013-01-02 00:00:00 to 2014-03-31 00:00:00 | rows: 454
Target missing: 0


### Feature engineering (PAST-ONLY)

In [4]:
# Lag features
df["lag_1"]  = df["unit_sales"].shift(1)
df["lag_7"]  = df["unit_sales"].shift(7)
df["lag_14"] = df["unit_sales"].shift(14)

# Rolling features (based only on past)
df["rolling_mean_7"] = df["unit_sales"].shift(1).rolling(7).mean()
df["rolling_std_7"]  = df["unit_sales"].shift(1).rolling(7).std()

In [5]:
# Calendar features (numeric)
df["day_of_week"] = df.index.dayofweek  # 0=Mon ... 6=Sun
df["month"]       = df.index.month
df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)

In [6]:
# Drop initial NaNs from lags/rolling
min_required = ["lag_1", "lag_7", "lag_14", "rolling_mean_7", "rolling_std_7"]
df_ml = df.dropna(subset=min_required).copy()

print("Rows after lag/rolling cleanup:", len(df_ml), "| starts:", df_ml.index.min())

Rows after lag/rolling cleanup: 440 | starts: 2013-01-16 00:00:00


### Time split

In [7]:
train_ml = df_ml.loc["2013-01-02":"2013-12-31"].copy()
test_ml  = df_ml.loc["2014-01-01":"2014-03-31"].copy()

print("Train:", train_ml.index.min(), "to", train_ml.index.max(), "| rows:", len(train_ml))
print("Test :", test_ml.index.min(),  "to", test_ml.index.max(),  "| rows:", len(test_ml))

Train: 2013-01-16 00:00:00 to 2013-12-31 00:00:00 | rows: 350
Test : 2014-01-01 00:00:00 to 2014-03-31 00:00:00 | rows: 90


### Feature list (numeric-only, safe for sklearn)

In [8]:
features = [
    # lag & rolling
    "lag_1", "lag_7", "lag_14",
    "rolling_mean_7", "rolling_std_7",

    # calendar
    "day_of_week", "month", "is_weekend",

    # holiday flags
    "is_holiday", "is_national_holiday",
    "before_1", "before_2", "after_1",

    # weather
    "ALLSKY_SFC_SW_DWN",
    "PRECTOTCORR",
    "T2M", "T2M_MAX", "T2M_MIN",
    "is_sunny", "is_rainy",
    "solar_7d",

    # macro / oil
    "dcoilwtico",
    "oil_lag_7",
    "cpi",
    "min_wage",
]

missing = [c for c in features if c not in df_ml.columns]
if missing:
    raise ValueError(f"Missing feature columns in df_ml: {missing}")

# Matrices
X_train = train_ml[features]
y_train = train_ml["unit_sales"]

X_test  = test_ml[features]
y_test  = test_ml["unit_sales"]

print("X_train:", X_train.shape, "| X_test:", X_test.shape)

X_train: (350, 25) | X_test: (90, 25)


In [9]:
## check
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("Train range:", X_train.index.min(), "->", X_train.index.max())
print("Test range :", X_test.index.min(), "->", X_test.index.max())

print("Number of features:", len(X_train.columns))

X_train shape: (350, 25)
X_test shape: (90, 25)
Train range: 2013-01-16 00:00:00 -> 2013-12-31 00:00:00
Test range : 2014-01-01 00:00:00 -> 2014-03-31 00:00:00
Number of features: 25


In [10]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

baseline = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror"
)

baseline.fit(X_train, y_train)
pred = baseline.predict(X_test)

print("Baseline MAE:", mean_absolute_error(y_test, pred))

Baseline MAE: 111.52606913248698


### Metric helpers

In [11]:
def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) + 1e-8
    return float(100.0 * np.mean(2.0 * np.abs(y_pred - y_true) / denom))

def eval_regression(name: str, y_true: pd.Series, y_pred: np.ndarray) -> dict:
    mae  = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    s    = smape(y_true.values, y_pred)
    return {"Model": name, "MAE": mae, "RMSE": rmse, "sMAPE": s}

results = []

### Linear Regression

In [12]:
lr = LinearRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

res_lr = eval_regression("Linear Regression", y_test, pred_lr)
results.append(res_lr)

print("\nLinear Regression")
print("MAE :", res_lr["MAE"])
print("RMSE:", res_lr["RMSE"])
print("sMAPE:", res_lr["sMAPE"])


Linear Regression
MAE : 129.55761012281715
RMSE: 182.36405165527398
sMAPE: 31.013209986014367


### Random Forest

In [13]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

res_rf = eval_regression("Random Forest", y_test, pred_rf)
results.append(res_rf)

print("\nRandom Forest")
print("MAE :", res_rf["MAE"])
print("RMSE:", res_rf["RMSE"])
print("sMAPE:", res_rf["sMAPE"])


Random Forest
MAE : 106.33151851851851
RMSE: 152.84440894033705
sMAPE: 23.681669067796633


### XGBoost

In [14]:
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
xgb.fit(X_train, y_train)
pred_xgb = xgb.predict(X_test)

res_xgb = eval_regression("XGBoost", y_test, pred_xgb)
results.append(res_xgb)

print("\nXGBoost")
print("MAE :", res_xgb["MAE"])
print("RMSE:", res_xgb["RMSE"])
print("sMAPE:", res_xgb["sMAPE"])


XGBoost
MAE : 111.52606913248698
RMSE: 157.44557306574214
sMAPE: 24.784602944670638


#### Feature-engineered machine learning models achieve competitive performance compared to classical statistical models. Linear regression performs best among the ML approaches in this baseline setup, while Random Forest and XGBoost show similar but slightly weaker performance. This suggests that the underlying time series structure is largely linear with strong seasonal patterns.

In [15]:
## Save results table
import pandas as pd
from pathlib import Path

# Convert results list to dataframe
metrics_ml = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)

# Define models folder
ROOT = Path.cwd().parents[0]
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

# Save ML metrics
metrics_ml.to_csv(MODELS_DIR / "metrics_ml.csv", index=False)

print("Saved:", MODELS_DIR / "metrics_ml.csv")
metrics_ml

Saved: D:\data analytics\TimeSeries\models\metrics_ml.csv


,Model,MAE,RMSE,sMAPE
0,Random Forest,106.331519,152.844409,23.681669
1,XGBoost,111.526069,157.445573,24.784603
2,Linear Regression,129.557610,182.364052,31.013210
